# Notebook 01: Cygnus X-1 UTAC System Overview

**GenesisAeon Package 17** — `cygnus-jet-utac`

Reference: Prabu et al. (2026, Nature Astronomy, DOI: 10.1038/s41550-026-02828-3)

This notebook provides a system overview and demonstrates the calibrated UTAC parameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rich.console import Console

from cygnus_jet_utac import CygnusJetUTAC, __version__
from cygnus_jet_utac.constants import (
    CYG_BH_MASS, CYG_COMPANION_MASS, CYG_ORBITAL_PERIOD,
    CYG_JET_POWER, CYG_JET_VELOCITY, CYG_JET_EXTENT,
    M_SUN, L_SUN, LIGHT_YEAR
)
from cygnus_jet_utac.efficiency import calibrate_gamma_jet

np.random.seed(42)
print(f'cygnus-jet-utac v{__version__}')

## 1. System Parameters

In [ ]:
print('=== Cygnus X-1 System Parameters ===')
print(f'  Black hole mass:      {CYG_BH_MASS/M_SUN:.1f} M☉')
print(f'  Companion mass:       {CYG_COMPANION_MASS/M_SUN:.1f} M☉')
print(f'  Orbital period:       {CYG_ORBITAL_PERIOD/86400:.4f} days')
print(f'  Jet power:            {CYG_JET_POWER/L_SUN:,.0f} L☉ = {CYG_JET_POWER:.3e} W')
print(f'  Jet velocity:         {CYG_JET_VELOCITY/3e8:.2f} c')
print(f'  Jet extent:           {CYG_JET_EXTENT/LIGHT_YEAR:.1f} ly')

## 2. Γ_jet Calibration — Key Scientific Result

In [ ]:
result = calibrate_gamma_jet(eta=0.10, verbose=True)
print(result['report'])

## 3. UTAC Fixed-Point Landscape

In [ ]:
from cygnus_jet_utac.efficiency import gamma_scan

eta_arr, gamma_arr = gamma_scan()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(eta_arr, gamma_arr, 'b-', lw=2)
ax1.axvline(0.10, color='red', ls='--', label='η = 10% (Prabu 2026)')
ax1.axhline(result['gamma_jet'], color='orange', ls='--', label=f'Γ_jet = {result["gamma_jet"]:.4f}')
ax1.scatter([0.10], [result['gamma_jet']], color='red', zorder=5, s=80)
ax1.set_xlabel('Accretion-to-jet efficiency η')
ax1.set_ylabel('CREP Γ_jet')
ax1.set_title('η → Γ_jet Inversion (UTAC Fixed Point)')
ax1.legend()
ax1.grid(alpha=0.3)

gamma_fine = np.linspace(0, 2, 300)
H_star = np.tanh(2.2 * gamma_fine)
ax2.plot(gamma_fine, H_star, 'b-', lw=2)
ax2.axvline(result['gamma_jet'], color='red', ls='--', label=f'Γ_jet = {result["gamma_jet"]:.4f}')
ax2.axhline(0.10, color='orange', ls='--', label='H*/K = 0.10')
ax2.scatter([result['gamma_jet']], [0.10], color='red', zorder=5, s=80)
ax2.set_xlabel('CREP Γ')
ax2.set_ylabel('H*/K = tanh(σ·Γ)')
ax2.set_title('UTAC Fixed-Point Landscape')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Cygnus X-1 UTAC Calibration — GenesisAeon Package 17', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/utac_calibration.png', dpi=150)
plt.show()
print('Saved: docs/utac_calibration.png')

## 4. Initialise and Run a Short Simulation

In [ ]:
system = CygnusJetUTAC(dt=21600.0, seed=42)  # 6-hour steps
print(system)

# Run 2-year overview simulation
results = system.run_cycle(duration_years=2.0)

print(f'\n=== 2-Year Simulation Summary ===')
print(f'  Γ_jet:           {results["gamma_jet"]:.4f}')
print(f'  Jet power:       {results["jet_power_W"]:.3e} W')
print(f'  Jet velocity:    {results["jet_velocity_c"]:.2f} c')
print(f'  Dance events:    {results["n_dance_events"]} ({results["dance_events_per_year"]:.1f}/yr)')
print(f'  Frame Principle: {"✓" if results["frame_principle_satisfied"] else "✗"}')

In [ ]:
# CREP and UTAC state snapshots
crep = system.get_crep_state()
utac = system.get_utac_state()

print('CREP State:', {k: f'{v:.4f}' for k, v in crep.items()})
print('UTAC State:', {k: f'{v:.4f}' for k, v in utac.items()})